### All MySQL commands using Python for automating MySQL

In [1]:
import mysql.connector

DB_CONFIG = dict(host="localhost", user="root", password="garvit@123", database="aegis_neo")

def get_conn():
    return mysql.connector.connect(**DB_CONFIG)

def run(sql, description=""):
    conn = get_conn()
    cur = conn.cursor()
    try:
        cur.execute(sql)
        conn.commit()
        print(f"✅ {description or 'OK'} | rows affected: {cur.rowcount}")
    except Exception as e:
        print(f"❌ {description}: {e}")
    finally:
        cur.close(); conn.close()

def fetch(sql):
    conn = get_conn()
    cur = conn.cursor()
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [d[0] for d in cur.description]
    cur.close(); conn.close()
    return cols, rows

In [2]:
run("""
CREATE TABLE IF NOT EXISTS dim_neo (
    neo_id VARCHAR(20) PRIMARY KEY,
    full_name VARCHAR(150),
    is_hazardous BOOLEAN,
    diameter_km_avg FLOAT,
    absolute_magnitude_h FLOAT,
    eccentricity FLOAT,
    semi_major_axis_au FLOAT,
    inclination_deg FLOAT,
    orbital_period_days FLOAT,
    data_arc_days INT,
    n_observations INT,
    impact_probability FLOAT,
    palermo_scale_max FLOAT,
    torino_scale INT,
    last_obs_date VARCHAR(20)
)
""", "create dim_neo")

run("""
CREATE TABLE IF NOT EXISTS fact_close_approach (
    approach_id INT AUTO_INCREMENT PRIMARY KEY,
    neo_id VARCHAR(20),
    close_approach_date DATE,
    relative_velocity_kmh FLOAT,
    miss_distance_km FLOAT,
    miss_distance_ld FLOAT,
    orbiting_body VARCHAR(50),
    FOREIGN KEY (neo_id) REFERENCES dim_neo(neo_id)
)
""", "create fact_close_approach")

✅ create dim_neo | rows affected: 0
✅ create fact_close_approach | rows affected: 0


In [5]:
run("""
INSERT INTO dim_neo (neo_id, full_name, diameter_km_avg, absolute_magnitude_h,
    eccentricity, semi_major_axis_au, inclination_deg, orbital_period_days,
    data_arc_days, n_observations, impact_probability, palermo_scale_max,
    torino_scale, last_obs_date, is_hazardous)
SELECT
    o.neo_id, TRIM(o.full_name),
    AVG((f.est_diameter_min_km + f.est_diameter_max_km) / 2),
    NULL, o.eccentricity, o.semi_major_axis_au, o.inclination_deg,
    o.orbital_period_days, o.data_arc_days, o.n_observations,
    s.impact_probability, s.palermo_scale_max, s.torino_scale,
    s.last_obs_date, MAX(f.is_hazardous)
FROM raw_orbital_elements o
LEFT JOIN raw_sentry_risk s ON o.neo_id = s.neo_id
LEFT JOIN raw_neo_feed f ON o.neo_id = f.neo_id
GROUP BY o.neo_id, o.full_name, o.eccentricity, o.semi_major_axis_au,
         o.inclination_deg, o.orbital_period_days, o.data_arc_days,
         o.n_observations, s.impact_probability, s.palermo_scale_max,
         s.torino_scale, s.last_obs_date
""", "populate dim_neo")

✅ populate dim_neo | rows affected: 2000


In [7]:
run("""
INSERT INTO fact_close_approach (neo_id, close_approach_date, relative_velocity_kmh,
    miss_distance_km, miss_distance_ld, orbiting_body)
SELECT f.neo_id, f.close_approach_date, f.relative_velocity_kmh, f.miss_distance_km,
    f.miss_distance_km / 384400, f.orbiting_body
FROM raw_neo_feed f
INNER JOIN dim_neo d ON f.neo_id = d.neo_id
""", "populate fact_close_approach")

✅ populate fact_close_approach | rows affected: 0


In [8]:
cols, rows = fetch("SELECT COUNT(*) FROM raw_neo_feed")
print("raw_neo_feed total:", rows[0][0])
cols, rows = fetch("SELECT COUNT(DISTINCT neo_id) FROM raw_neo_feed")
print("raw_neo_feed unique neo_id:", rows[0][0])
cols, rows = fetch("SELECT COUNT(*) FROM fact_close_approach")
print("fact_close_approach inserted:", rows[0][0])


raw_neo_feed total: 40
raw_neo_feed unique neo_id: 40
fact_close_approach inserted: 0


In [10]:
import pandas as pd

In [11]:
cols, rows = fetch("SELECT neo_id, name FROM raw_neo_feed LIMIT 5")
print(pd.DataFrame(rows, columns=cols))

cols, rows = fetch("SELECT neo_id, full_name FROM dim_neo LIMIT 5")
print(pd.DataFrame(rows, columns=cols))

    neo_id                name
0  2240320  240320 (2003 HS42)
1  2250680   250680 (2005 QC5)
2  2452639   452639 (2005 UY6)
3  2500136  500136 (2012 CO46)
4  2523808  523808 (2007 ML24)
     neo_id               full_name
0  20000433      433 Eros (A898 PA)
1  20000719    719 Albert (A911 TB)
2  20000887    887 Alinda (A918 AA)
3  20001036  1036 Ganymed (A924 UB)
4  20001221    1221 Amor (1932 EA1)


In [12]:
run("ALTER TABLE raw_neo_feed ADD COLUMN designation_num VARCHAR(20)", "add designation_num to raw_neo_feed")
run("ALTER TABLE dim_neo ADD COLUMN designation_num VARCHAR(20)", "add designation_num to dim_neo")

run("""
UPDATE raw_neo_feed
SET designation_num = TRIM(SUBSTRING_INDEX(TRIM(name), ' ', 1))
""", "populate designation_num in raw_neo_feed")

run("""
UPDATE dim_neo
SET designation_num = TRIM(SUBSTRING_INDEX(TRIM(full_name), ' ', 1))
""", "populate designation_num in dim_neo")

✅ add designation_num to raw_neo_feed | rows affected: 0
✅ add designation_num to dim_neo | rows affected: 0
✅ populate designation_num in raw_neo_feed | rows affected: 40
✅ populate designation_num in dim_neo | rows affected: 2000


In [13]:
cols, rows = fetch("""
SELECT COUNT(*) FROM raw_neo_feed f
INNER JOIN dim_neo d ON f.designation_num = d.designation_num
""")
print("matched rows:", rows[0][0])

matched rows: 2


In [ ]:
import requests
import mysql.connector

API_KEY = "hBnrFPk07qCcnQy1unyYO2VDgD1nAKH9gi8liBHu"
DB_CONFIG = dict(host="localhost", user="root", password="yourpass", database="aegis_neo")

def get_conn():
    return mysql.connector.connect(**DB_CONFIG)

def fetch_neo_feed(start_date, end_date):
    url = f"https://api.nasa.gov/neo/rest/v1/feed?start_date={start_date}&end_date={end_date}&api_key={API_KEY}"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.json()["near_earth_objects"]

def insert_feed_data(data):
    conn = get_conn()
    cur = conn.cursor()
    for date, objects in data.items():
        for obj in objects:
            approach = obj["close_approach_data"][0]
            cur.execute("""
                INSERT IGNORE INTO raw_neo_feed
                (neo_id, name, absolute_magnitude_h, est_diameter_min_km, est_diameter_max_km,
                 is_hazardous, close_approach_date, relative_velocity_kmh, miss_distance_km, orbiting_body)
                VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
            """, (
                obj["id"], obj["name"], obj["absolute_magnitude_h"],
                obj["estimated_diameter"]["kilometers"]["estimated_diameter_min"],
                obj["estimated_diameter"]["kilometers"]["estimated_diameter_max"],
                obj["is_potentially_hazardous_asteroid"], date,
                float(approach["relative_velocity"]["kilometers_per_hour"]),
                float(approach["miss_distance"]["kilometers"]),
                approach["orbiting_body"]
            ))
    conn.commit()
    cur.close(); conn.close()

SyntaxError: unterminated string literal (detected at line 4) (2963432128.py, line 4)

In [16]:
import time
from datetime import datetime, timedelta

today = datetime.today()
for i in range(0, 180, 7):  # ~6 months, week by week
    start = (today - timedelta(days=i+7)).strftime("%Y-%m-%d")
    end = (today - timedelta(days=i)).strftime("%Y-%m-%d")
    try:
        insert_feed_data(fetch_neo_feed(start, end))
        print(f"✅ {start} to {end}")
    except Exception as e:
        print(f"❌ {start} to {end}: {e}")
    time.sleep(1)

❌ 2026-07-04 to 2026-07-11: 403 Client Error: Forbidden for url: https://api.nasa.gov/neo/rest/v1/feed?start_date=2026-07-04&end_date=2026-07-11&api_key=YOUR_NASA_KEY
❌ 2026-06-27 to 2026-07-04: 403 Client Error: Forbidden for url: https://api.nasa.gov/neo/rest/v1/feed?start_date=2026-06-27&end_date=2026-07-04&api_key=YOUR_NASA_KEY
❌ 2026-06-20 to 2026-06-27: 403 Client Error: Forbidden for url: https://api.nasa.gov/neo/rest/v1/feed?start_date=2026-06-20&end_date=2026-06-27&api_key=YOUR_NASA_KEY
❌ 2026-06-13 to 2026-06-20: 403 Client Error: Forbidden for url: https://api.nasa.gov/neo/rest/v1/feed?start_date=2026-06-13&end_date=2026-06-20&api_key=YOUR_NASA_KEY
❌ 2026-06-06 to 2026-06-13: 403 Client Error: Forbidden for url: https://api.nasa.gov/neo/rest/v1/feed?start_date=2026-06-06&end_date=2026-06-13&api_key=YOUR_NASA_KEY
❌ 2026-05-30 to 2026-06-06: 403 Client Error: Forbidden for url: https://api.nasa.gov/neo/rest/v1/feed?start_date=2026-05-30&end_date=2026-06-06&api_key=YOUR_NASA_KE